# Student Clustering Analysis

KMeans + DBSCAN hybrid clustering for student archetype discovery.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

In [ ]:
data_path = os.path.join("..", "..", "data_pipeline", "data", "students.json")
with open(data_path, "r") as f:
    students_raw = json.load(f)

df = pd.json_normalize(students_raw)

subjects = [
    "DSA", "OOP", "DBMS", "OS", "Networks",
    "Mathematics", "Physics", "English", "Machine Learning", "Web Development"
]
slots = ["morning", "afternoon", "evening"]
pace_map = {"slow": 0.0, "moderate": 0.5, "fast": 1.0}

feature_rows = []
for student in students_raw:
    row = []
    row.append(student["cgpa"] / 10.0)
    row.append(student["year"] / 4.0)
    row.append(pace_map[student["learning_pace"]])

    skill_lookup = {s["subject"]: s for s in student["skills"]}
    skill_vals = []
    for subj in subjects:
        s = skill_lookup.get(subj, {})
        sr = s.get("self_rating", 5.0) / 10.0
        pr = s.get("peer_rating", 5.0) / 10.0
        gp = s.get("grade_points", 5.0) / 10.0
        row.extend([sr, pr, gp])
        skill_vals.append((sr + pr + gp) / 3.0)

    row.append(np.mean(skill_vals))
    row.append(np.std(skill_vals))

    for entry in student["availability"]:
        row.append(1.0 if entry["is_available"] else 0.0)

    feature_rows.append(row)

X = np.array(feature_rows)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Feature matrix shape: {X_scaled.shape}")

## 1. KMeans - Elbow Method

In [ ]:
k_range = range(2, 16)
inertias = []

for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(list(k_range), inertias, "bo-", linewidth=2, markersize=8)
ax.set_xlabel("Number of Clusters (k)", fontsize=12)
ax.set_ylabel("Inertia", fontsize=12)
ax.set_title("Elbow Method for Optimal k", fontsize=14, fontweight="bold")
ax.set_xticks(list(k_range))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Silhouette Score Analysis

In [ ]:
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    sil_scores.append(score)
    print(f"k={k:2d}  silhouette={score:.4f}")

best_k = list(k_range)[np.argmax(sil_scores)]
print(f"\nBest k by silhouette score: {best_k} (score={max(sil_scores):.4f})")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(list(k_range), sil_scores, "gs-", linewidth=2, markersize=8)
ax.axvline(best_k, color="red", linestyle="--", label=f"Best k={best_k}")
ax.set_xlabel("Number of Clusters (k)", fontsize=12)
ax.set_ylabel("Silhouette Score", fontsize=12)
ax.set_title("Silhouette Score vs. k", fontsize=14, fontweight="bold")
ax.set_xticks(list(k_range))
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. KMeans Clustering (k=8)

In [ ]:
kmeans = KMeans(n_clusters=8, n_init=10, random_state=42)
km_labels = kmeans.fit_predict(X_scaled)

print("Cluster distribution (KMeans k=8):")
unique, counts = np.unique(km_labels, return_counts=True)
for cluster_id, count in zip(unique, counts):
    print(f"  Cluster {cluster_id}: {count} students ({count / len(km_labels) * 100:.1f}%)")

print(f"\nSilhouette score: {silhouette_score(X_scaled, km_labels):.4f}")

## 4. PCA Visualization

In [ ]:
pca_2d = PCA(n_components=2, random_state=42)
X_2d = pca_2d.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(
    X_2d[:, 0], X_2d[:, 1],
    c=km_labels, cmap="tab10", alpha=0.6, s=30, edgecolors="k", linewidths=0.3
)

centroids_2d = pca_2d.transform(kmeans.cluster_centers_)
ax.scatter(
    centroids_2d[:, 0], centroids_2d[:, 1],
    c="red", marker="X", s=200, edgecolors="black", linewidths=2, zorder=5, label="Centroids"
)

ax.set_xlabel(f"PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% var)", fontsize=12)
ax.set_ylabel(f"PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% var)", fontsize=12)
ax.set_title("KMeans Clusters (PCA 2D Projection)", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
plt.colorbar(scatter, ax=ax, label="Cluster ID")
plt.tight_layout()
plt.show()

## 5. DBSCAN Outlier Detection

In [ ]:
pca_20 = PCA(n_components=20, random_state=42)
X_pca20 = pca_20.fit_transform(X_scaled)
print(f"PCA 20-dim explains {pca_20.explained_variance_ratio_.sum()*100:.1f}% of variance")

dbscan = DBSCAN(eps=0.5, min_samples=3)
db_labels = dbscan.fit_predict(X_pca20)

n_outliers = np.sum(db_labels == -1)
n_clusters_db = len(set(db_labels) - {-1})
print(f"DBSCAN found {n_clusters_db} clusters and {n_outliers} outliers ({n_outliers/len(db_labels)*100:.1f}%)")

X_2d_all = pca_2d.transform(X_scaled)
outlier_mask = db_labels == -1

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(
    X_2d_all[~outlier_mask, 0], X_2d_all[~outlier_mask, 1],
    c="steelblue", alpha=0.4, s=25, label=f"Normal ({np.sum(~outlier_mask)})"
)
ax.scatter(
    X_2d_all[outlier_mask, 0], X_2d_all[outlier_mask, 1],
    c="red", alpha=0.8, s=50, marker="x", linewidths=2, label=f"Outliers ({n_outliers})"
)
ax.set_xlabel("PC1", fontsize=12)
ax.set_ylabel("PC2", fontsize=12)
ax.set_title("DBSCAN Outlier Detection (PCA 2D)", fontsize=14, fontweight="bold")
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

## 6. Hybrid: Reassign Outliers

In [ ]:
hybrid_labels = km_labels.copy()

outlier_indices = np.where(outlier_mask)[0]
if len(outlier_indices) > 0:
    outlier_features = X_scaled[outlier_indices]
    distances = np.linalg.norm(
        outlier_features[:, np.newaxis, :] - kmeans.cluster_centers_[np.newaxis, :, :],
        axis=2
    )
    nearest_clusters = np.argmin(distances, axis=1)
    hybrid_labels[outlier_indices] = nearest_clusters
    print(f"Reassigned {len(outlier_indices)} DBSCAN outliers to nearest KMeans centroids")
else:
    print("No outliers to reassign")

hybrid_sil = silhouette_score(X_scaled, hybrid_labels)
km_sil = silhouette_score(X_scaled, km_labels)
print(f"\nKMeans-only silhouette:  {km_sil:.4f}")
print(f"Hybrid silhouette:       {hybrid_sil:.4f}")

print("\nFinal cluster distribution:")
unique, counts = np.unique(hybrid_labels, return_counts=True)
for cid, cnt in zip(unique, counts):
    print(f"  Cluster {cid}: {cnt} students")

## 7. Cluster Profiles

In [ ]:
df["cluster"] = hybrid_labels

skill_means_per_student = []
for student in students_raw:
    ratings = [s["self_rating"] for s in student["skills"]]
    skill_means_per_student.append(np.mean(ratings))
df["mean_skill_rating"] = skill_means_per_student

print("=== Cluster Profiles ===")
print(f"{'Cluster':>8} {'Size':>6} {'Mean CGPA':>10} {'Mean Skill':>11} {'Top Dept':>10}")
print("-" * 50)

for cid in sorted(df["cluster"].unique()):
    subset = df[df["cluster"] == cid]
    mean_cgpa = subset["cgpa"].mean()
    mean_skill = subset["mean_skill_rating"].mean()
    top_dept = subset["department"].value_counts().index[0]
    print(f"{cid:>8} {len(subset):>6} {mean_cgpa:>10.2f} {mean_skill:>11.2f} {top_dept:>10}")